<a href="https://colab.research.google.com/github/Gauravd1710/legal-doc-analyzer/blob/main/notebooks/02_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

BASE      = '/content/drive/MyDrive/LegalDocAnalyzer'
REPO_PATH = '/content/legal-doc-analyzer'

sys.path.append(f'{REPO_PATH}/src')

print("✅ Drive mounted")
print(f"BASE      : {BASE}")
print(f"REPO_PATH : {REPO_PATH}")

In [ ]:
# datasets    → download CUAD & LEDGAR from HuggingFace Hub
# pandas      → inspect and filter data in table form
# sklearn     → train/val/test splitting

!pip install datasets pandas scikit-learn -q

from datasets import load_dataset
import pandas as pd
import json, os
from sklearn.model_selection import train_test_split

print("✅ Libraries ready")

In [ ]:
import json

with open(f'{BASE}/data/raw/CUAD_v1.json') as f:
    raw = json.load(f)

# CUAD is in SQuAD format — data → paragraphs → qas
rows = []
for article in raw['data']:
    title = article['title']
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            rows.append({
                'title'   : title,
                'question': qa['id'].split('__')[-1].replace('_', ' '),
                'context' : context,
                'answers' : {
                    'text'        : [a['text'] for a in qa['answers']],
                    'answer_start': [a['answer_start'] for a in qa['answers']]
                }
            })

import pandas as pd
df_cuad = pd.DataFrame(rows)

print(f"✅ Loaded CUAD manually")
print(f"Total rows : {len(df_cuad)}")
print(f"\nColumns: {df_cuad.columns.tolist()}")
print(f"\nSample:")
print(df_cuad.iloc[0])

In [ ]:
# verify the structure looks right

print("=== SHAPE ===")
print(f"Rows: {len(df_cuad)}, Columns: {df_cuad.columns.tolist()}")

print("\n=== SAMPLE ANSWER STRUCTURE ===")
sample = df_cuad.iloc[0]
print(f"Title    : {sample['title'][:60]}")
print(f"Question : {sample['question']}")
print(f"Context  : {sample['context'][:200]}")
print(f"Answer   : {sample['answers']}")

print("\n=== NULL CHECK ===")
print(df_cuad.isnull().sum())

print("\n=== ANSWER COUNT ===")
df_cuad['has_answer'] = df_cuad['answers'].apply(
    lambda x: len(x['text']) > 0
)
print(f"Rows WITH answers    : {df_cuad['has_answer'].sum()}")
print(f"Rows WITHOUT answers : {(~df_cuad['has_answer']).sum()}")
print(f"Usable rows          : {df_cuad['has_answer'].sum()}")

In [ ]:
# WHY: Need to see exact question strings in THIS version
# of CUAD so our keyword mapping in Cell 6 matches correctly

questions = df_cuad['question'].unique()
print(f"Total unique question types: {len(questions)}\n")
for i, q in enumerate(questions):
    print(f"{i+1:02d}. {q}")

In [ ]:
ENTITY_MAP = {
    "PARTY": [
        "parties", "licensor", "licensee",
        "counterparty", "document name",
        "license grant"                      # ← rescued
    ],
    "DATE": [
        "effective date", "expiration date",
        "renewal term", "notice period",
        "agreement date"
    ],
    "AMOUNT": [
        "revenue", "payment", "price", "fee",
        "cap on liability", "minimum commitment",
        "volume restriction", "post-termination",
        "uncapped liability",                 # ← rescued
        "liquidated damages",                 # ← rescued
        "most favored nation"                 # ← rescued
    ],
    "JURISDICTION": [
        "governing law", "jurisdiction"
    ],
    "TERM": [
        "termination", "indemnification", "confidentiality",
        "non-compete", "ip ownership", "warranty",
        "non-disparagement", "audit rights",
        "exclusivity",                        # ← rescued
        "covenant not to sue",                # ← rescued
        "no-solicit of employees",            # ← rescued
        "no-solicit of customers",            # ← rescued
        "competitive restriction"             # ← rescued
    ]
}

df_useful = df_cuad[df_cuad['has_answer']].copy()

print("=== UPDATED ENTITY MAPPING ===\n")
total_matched = 0

for entity, keywords in ENTITY_MAP.items():
    mask = df_useful['question'].str.lower().apply(
        lambda q: any(kw in q.lower() for kw in keywords)
    )
    count = mask.sum()
    total_matched += count
    print(f"{entity:15s} → {count:5d} rows")

print(f"\nTotal mapped rows : {total_matched}")
print(f"Unmapped rows     : {len(df_useful) - total_matched}")

In [ ]:
def assign_entity_label(question):
    q = question.lower()
    for entity, keywords in ENTITY_MAP.items():
        if any(kw in q for kw in keywords):
            return entity
    return None

df_useful['entity_label'] = df_useful['question'].apply(assign_entity_label)

before = len(df_useful)
df_useful = df_useful[df_useful['entity_label'].notna()].copy()
after = len(df_useful)

print("=== UPDATED LABEL DISTRIBUTION ===\n")
counts = df_useful['entity_label'].value_counts()
print(counts)

total = counts.sum()
print("\n% distribution:")
for label, count in counts.items():
    bar = "█" * int(count / total * 40)
    print(f"{label:15s} {count:5d}  {bar}")

min_count = counts.min()
max_count = counts.max()
ratio = max_count / min_count

print(f"\nRows kept      : {after}")
print(f"Rows dropped   : {before - after}")
print(f"Imbalance ratio: {ratio:.1f}x  ", end="")
if ratio > 3:
    print("⚠️  Still imbalanced — will handle in Step 3")
else:
    print("✅ Balanced enough")

In [ ]:
# WHY: Sanity check — make sure each entity label
# has sensible context text and answer text

print("=== ONE SAMPLE PER ENTITY TYPE ===\n")

for entity in ENTITY_MAP.keys():
    sample = df_useful[df_useful['entity_label'] == entity].iloc[0]
    print(f"{'='*50}")
    print(f"ENTITY    : {entity}")
    print(f"QUESTION  : {sample['question']}")
    print(f"ANSWER    : {sample['answers']['text'][0][:100]}")
    print(f"CONTEXT   : {sample['context'][:150]}")
    print()

##LEDGAR

In [ ]:
# WHY LEDGAR: 60k pre-segmented clauses
# Each row IS one complete clause → perfect for
# training the clause segmenter
# No annotation needed — structure itself is the label

from datasets import load_dataset

print("Downloading LEDGAR...")

ledgar = load_dataset(
    "lex_glue", "ledgar",
    verification_mode='no_checks'
)

print(ledgar)
print(f"\nTrain rows      : {len(ledgar['train'])}")
print(f"Validation rows : {len(ledgar['validation'])}")
print(f"Test rows       : {len(ledgar['test'])}")

In [ ]:
#ledgar structure
sample = ledgar['train'][0]
print("=== ONE LEDGAR SAMPLE ===")
print(f"Label : {sample['label']}")
print(f"Text  : {sample['text'][:400]}")

print("\n=== ALL LEDGAR CLAUSE TYPES ===")
label_names = ledgar['train'].features['label'].names
for i, name in enumerate(label_names):
    print(f"{i:03d}. {name}")

In [ ]:
# WHY: Sets the thresholds for your clause segmenter
# Clauses too short = fragments, too long = needs splitting
# This chart tells you where to draw the line

import matplotlib.pyplot as plt

df_ledgar = pd.DataFrame(ledgar['train'])
df_ledgar['word_count'] = df_ledgar['text'].apply(lambda x: len(x.split()))

print("=== CLAUSE LENGTH STATS ===")
print(df_ledgar['word_count'].describe())

plt.figure(figsize=(10, 4))
plt.hist(df_ledgar['word_count'], bins=50,
         color='steelblue', edgecolor='white')
plt.title('LEDGAR Clause Length Distribution')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.axvline(10,  color='red',   linestyle='--', label='Min threshold (10)')
plt.axvline(300, color='green', linestyle='--', label='Max threshold (300)')
plt.legend()
plt.tight_layout()
plt.show()

short = (df_ledgar['word_count'] < 10).sum()
good  = ((df_ledgar['word_count'] >= 10) &
         (df_ledgar['word_count'] <= 300)).sum()
long_ = (df_ledgar['word_count'] > 300).sum()

print(f"\nToo short  (<10 words)  : {short}")
print(f"Good range (10-300)     : {good}")
print(f"Too long   (>300 words) : {long_}")

#Saving to drive

In [ ]:

# save CUAD mapped rows
cuad_path = f'{BASE}/data/raw/cuad_mapped.json'
df_useful.to_json(cuad_path, orient='records', indent=2)
print(f"✅ CUAD saved   : {len(df_useful)} rows → cuad_mapped.json")

# save LEDGAR
ledgar_path = f'{BASE}/data/raw/ledgar_train.json'
df_ledgar.to_json(ledgar_path, orient='records', indent=2)
print(f"✅ LEDGAR saved : {len(df_ledgar)} rows → ledgar_train.json")

In [ ]:
# WHY 3 splits not 2:
#   train      → model learns from this
#   validation → monitor during training, tune hyperparams
#   test       → FROZEN, only used for final evaluation

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_useful, test_size=0.30, random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42
)

print("=== CUAD SPLITS ===")
print(f"Train      : {len(train_df)} rows  (70%)")
print(f"Validation : {len(val_df)}  rows  (15%)")
print(f"Test       : {len(test_df)} rows  (15%)")
print(f"Total      : {len(train_df)+len(val_df)+len(test_df)}")

# save splits
train_df.to_json(f'{BASE}/data/processed/cuad_train.json', orient='records', indent=2)
val_df.to_json(f'{BASE}/data/processed/cuad_val.json',     orient='records', indent=2)
test_df.to_json(f'{BASE}/data/processed/cuad_test.json',   orient='records', indent=2)

print("\n✅ All 3 splits saved to Drive")
print("⚠️  DO NOT use cuad_test.json until final evaluation")

FINAL SUMMARY

In [ ]:
print("=" * 50)
print("       STEP 2 COMPLETE — DATASET SUMMARY")
print("=" * 50)

print(f"""
📦 CUAD
   Total rows loaded    : 20,910
   Rows with answers    : 6,702
   After entity mapping : 5,686
   Train split          : 3,980
   Validation split     :   853
   Test split (frozen)  :   853

   Label distribution:
   DATE            1560
   PARTY           1356
   TERM            1248
   AMOUNT          1085
   JURISDICTION     437
   Imbalance ratio : 3.6x  ← fix in Step 3

📦 LEDGAR
   Total clauses        : 60,000
   Good length (10-300) : 56,773
   Too short (<10)      :    130  ← drop in Step 3
   Too long  (>300)     :  3,097  ← split in Step 3

📁 Saved to Drive:
   data/raw/cuad_mapped.json
   data/raw/ledgar_train.json
   data/processed/cuad_train.json
   data/processed/cuad_val.json
   data/processed/cuad_test.json  ← FROZEN
""")

In [ ]:
import os, json
import pandas as pd

BASE = '/content/drive/MyDrive/LegalDocAnalyzer'

# ── save dataframes ──
df_useful.to_json(f'{BASE}/data/raw/cuad_mapped.json',
                  orient='records', indent=2)

df_cuad.to_json(f'{BASE}/data/raw/cuad_full.json',
                orient='records', indent=2)

df_ledgar.to_json(f'{BASE}/data/raw/ledgar_train.json',
                  orient='records', indent=2)

train_df.to_json(f'{BASE}/data/processed/cuad_train.json',
                 orient='records', indent=2)
val_df.to_json(f'{BASE}/data/processed/cuad_val.json',
               orient='records', indent=2)
test_df.to_json(f'{BASE}/data/processed/cuad_test.json',
                orient='records', indent=2)

# ── save entity map ──
with open(f'{BASE}/src/entity_map.json', 'w') as f:
    json.dump(ENTITY_MAP, f, indent=2)

print("✅ All session data saved to Drive")
print(f"\nFiles saved:")
print(f"  data/raw/cuad_mapped.json")
print(f"  data/raw/cuad_full.json")
print(f"  data/raw/ledgar_train.json")
print(f"  data/processed/cuad_train.json")
print(f"  data/processed/cuad_val.json")
print(f"  data/processed/cuad_test.json  ← FROZEN")
print(f"  src/entity_map.json")

In [ ]:
import shutil, os

# save session state so recovery is instant next time
df_useful.to_json(f'{BASE}/data/raw/cuad_mapped.json',
                  orient='records', indent=2)

with open(f'{BASE}/src/entity_map.json', 'w') as f:
    json.dump(ENTITY_MAP, f, indent=2)

print("✅ Session state saved to Drive")

# push to GitHub
os.chdir(REPO_PATH)
!git pull

shutil.copy(
    '/content/02_data_preparation.ipynb',
    f'{REPO_PATH}/notebooks/02_data_preparation.ipynb'
)

# copy entity_map to repo src
shutil.copy(
    f'{BASE}/src/entity_map.json',
    f'{REPO_PATH}/src/entity_map.json'
)

!git add .
!git commit -m "Step 2 complete: CUAD 5686 rows, LEDGAR 60k, 70/15/15 split saved"
!git push

print("✅ Pushed to GitHub")
